# 03. Modeling

## Prerequisites

This notebook models `market_value_in_eur_log1p` using the train, validation and test splits (`train.parquet`, `val.parquet` and `test.parquet`) built based on the findings in `02_eda_and_feature_engineering.ipynb`. Each row is one player at one valuation date, with the features found in the EDA notebook and the player's previous valuation attached.

The end goals are to find out how much of Transfermarkt's valuation can be reproduced from a player's profile and recent statistics, how much comes from the previous valuation alone, and what the best model relies on (using SHAP). Models are compared against simple baselines on the validation set, and the test set is used once, at the end.

### Import libraires

In [ ]:
from transferformance.configs import DIR_PROCESSED_SPLITS, DIR_MODEL
import pandas as pd
import numpy as np

from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor
import LightGBM as lgb
import shap
import optuna

from typing import Literal
import json
from pathlib import Path

In [69]:
optuna.logging.set_verbosity(optuna.logging.INFO)

### Prepare the splits

We load the training, validation and test sets we built earlier. They hold 178,658, 18,003 and 21,170 rows respectively (about 82%, 8% and 10% of the data).

In [70]:
df_train = pd.read_parquet(DIR_PROCESSED_SPLITS / "train.parquet")
df_val = pd.read_parquet(DIR_PROCESSED_SPLITS / "val.parquet")
df_test = pd.read_parquet(DIR_PROCESSED_SPLITS / "test.parquet")

print(f"Training set: {df_train.shape}")
print(f"Validation set: {df_val.shape}")
print(f"Test set: {df_test.shape}")

Training set: (178658, 28)
Validation set: (18003, 28)
Test set: (21170, 28)


### Define features

We define the target and the features. The target is `market_value_in_eur_log1p`. The features follow the EDA and are grouped into categorical (`FEATURES_CAT`), numerical (`FEATURES_NUM`) and previous-valuation features (`FEATURES_HISTORY`).

We combine them into two sets: 
- `FEATURES_WITHOUT_HISTORY` estimates a value from the player's profile and recent statistics alone.
- `FEATURES_WITH_HISTORY` also includes the previous valuation and the days since it.

In [71]:
TARGET = "market_value_in_eur_log1p"

FEATURES_CAT = ["position", "foot", "league_id"]

FEATURES_NUM = [
    "age",
    "height_in_cm",
    "n_appearances",
    "minutes_per_365d",
    "league_minutes_per_365d",
    "cup_minutes_per_365d",
    "avg_minutes_per_appearance",
    "goals_per_365d",
    "goals_per_90m",
    "assists_per_365d",
    "assists_per_90m",
    "yellow_cards_per_365d",
    "red_cards_per_365d",
    "goals_conceded_per_365d",
    "goals_conceded_per_90m",
    "clean_sheets_per_365d",
    "clean_sheets_per_90m",
]

FEATURES_HISTORY = ["previous_valuation", "days_since_previous_valuations"]

FEATURES_WITHOUT_HISTORY = FEATURES_CAT + FEATURES_NUM
FEATURES_WITH_HISTORY = FEATURES_WITHOUT_HISTORY + FEATURES_HISTORY

We set a random seed, used by every model with a random component, so the results can be reproduced.

In [72]:
SEED = 42

### Define metrics

We evaluate models using three core metrics, all calculated consistently across every data split. 

Since our targets are on a log scale, predictions are converted back to EUR whenever a percentage is needed.


| Name | Usage | Example |
| --- | --- | --- |
| **MedAPE %** | Shows typical relative error without letting a few massive misses skew the score, treating cheap and expensive players equally. | A 10% score means half of the predictions fall within 10% of the true valuation, whether the player is worth 200k EUR or 50M EUR. |
| **Log RMSE** | Captures the tail-end misses that MedAPE ignores by squaring errors on the log scale, while still avoiding bias toward expensive players. | A single massive valuation miss penalizes this score heavily, whereas MedAPE would shrug it off. |
| **Bias** | Catches whether a model systematically lowballs or overshoots, balancing out MedAPE's natural preference for under-predicting. | Under-predicting caps out at a 100% error, while over-predicting has no ceiling; a negative bias exposes a model that is risk-averse. |

We skip $R^2$ and MAE in EUR as headline metrics.
- MAE is sensitive to outliers, which are plentty in this dataset where prices ranges from tens of thousands of euros to upward of hundreds of millions of euros.
- When past-valuation-related features are used, $R^2$ almost always sit near 1, which prevents us from telling models apart.

In [73]:
def evaluate(split: Literal["train", "val", "test"], model_name, y, y_hat):
    if isinstance(y, lgb.Dataset):
        y = y.get_label()

    y = np.asarray(y)
    y_hat = np.asarray(y_hat)

    err = y_hat - y
    ape = np.abs(np.expm1(y_hat) - np.expm1(y)) / np.expm1(y)
    print(f"""
    Results (`{split}` split):
        Model name: {model_name}
        MedAPE %: {100 * np.median(ape):.2f}
        Log RMSE: {np.sqrt((err ** 2).mean()):.4f}
        Bias: {err.mean():.4f}
    """)

## Examine models

### Pure value prediction models

In this section, we predict the market value without the previous valuation. The models only see the player's profile and recent statistics (`FEATURES_WITHOUT_HISTORY`), so they show how much of Transfermarkt's valuation can be reproduced from the player alone.

##### Our baseline: ``DummyRegressor`` with `median` strategy.

Our baseline is a ``DummyRegressor`` with the `median` strategy. It predicts the median training value for every player and uses no features, so its error is what we get from knowing only the overall distribution of values. 

In [ ]:
%%time

dummy = DummyRegressor(strategy="median").fit(df_train[FEATURES_NUM], df_train[TARGET])

evaluate("val", "dummy-median", df_val[TARGET], dummy.predict(df_val[FEATURES_NUM]))


    Results (`val` split):
        Model name: dummy-median
        MedAPE %: 81.82
        Log RMSE: 1.6890
        Bias: -0.5933
    
CPU times: total: 31.2 ms
Wall time: 17.5 ms


**Observations**:

- The baseline is far off, with a MedAPE of approx. 82%, which means for half of the players the prediction misses by more than 80% of the true value.
- The log RMSE is approx. 1.69, a typical error of a factor of about 5 in EUR. This is about the spread of the target itself, so it is the number to beat.
- The bias is approx. -0.59, so the predictions are about 45% too low on average. This is expected for a median predictor on a right-skewed target, since the mean sits above the median, and some of it may also come from the validation seasons being more recent.

**Decision:** Use this as the floor. Every model has to beat:
- `MedAPE` of 81.82% and 
- `Log RMSE` of 1.689 

on the validation set.

#### ``GradientBoostingRegressor``

For comparison, we train scikit-learn's classic ``GradientBoostingRegressor`` with default settings. It has no native categorical support, so `position` and `foot` are one-hot encoded, and unseen categories in validation are ignored.

In [ ]:
gb_no_hist = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURES_CAT),
        remainder="passthrough",
    ),
    GradientBoostingRegressor(random_state=SEED),
)

In [76]:
%%time
gb_no_hist.fit(df_train[FEATURES_WITHOUT_HISTORY], df_train[TARGET])

evaluate(
    "val",
    "gb-no-history",
    df_val[TARGET],
    gb_no_hist.predict(df_val[FEATURES_WITHOUT_HISTORY]),
)


    Results (`val` split):
        Model name: gb-no-history
        MedAPE %: 52.63
        Log RMSE: 0.9271
        Bias: -0.3004
    
CPU times: total: 1min 42s
Wall time: 1min 53s


**Observations**:
- The model beats the baseline. The MedAPE falls from 81.82% to 52.63%, and the log RMSE from 1.689 to 0.927, about 45% lower.
- The bias goes from -0.59 to -0.300, so it still under-predicts, by about 26% on average.
- Half of the predictions miss by more than 53% of the true value.
- Fitting and scoring take about 102 seconds of CPU time (about 1 minute 53 seconds wall) on 178,658 rows with default settings.

#### ``HistGradientBoostingRegressor``

Next, scikit-learn's ``HistGradientBoostingRegressor`` with default settings. It groups each feature into at most 255 bins before searching for splits, which is designed to make training much faster on large datasets, and it handles categorical columns natively. The categorical columns (`position`, `foot`) are converted to pandas categories using the training categories, so an unseen category in validation becomes missing instead of raising an error.

In [ ]:
for cat in FEATURES_CAT:
    dtype = pd.CategoricalDtype(sorted(df_train[cat].dropna().unique()))
    df_train[cat] = df_train[cat].astype(dtype)
    df_val[cat] = df_val[cat].astype(dtype)


hgb_no_hist = HistGradientBoostingRegressor(
    categorical_features="from_dtype",
    random_state=SEED,
)

In [78]:
%%time

hgb_no_hist.fit(df_train[FEATURES_WITHOUT_HISTORY], df_train[TARGET])

evaluate(
    "val",
    "hgb-no-history",
    df_val[TARGET],
    hgb_no_hist.predict(df_val[FEATURES_WITHOUT_HISTORY]),
)


    Results (`val` split):
        Model name: hgb-no-history
        MedAPE %: 46.62
        Log RMSE: 0.8593
        Bias: -0.3033
    
CPU times: total: 9.05 s
Wall time: 2.67 s


- The model beats the baseline. The MedAPE falls from 81.82% to 46.62%, and the log RMSE from 1.689 to 0.859, about 49% lower.
- Compared with `GradientBoostingRegressor`, it is better on the error metrics, with almost the same bias:
    - MedAPE: 46.62% against 52.63%
    - Log RMSE: 0.859 against 0.927 (about 7% lower)
    - Bias: -0.303 against -0.300
- Fitting and scoring take about 9 seconds of CPU time (2.7 seconds wall), against about 102 seconds of CPU time (113 seconds wall) for `GradientBoostingRegressor`. That is about 11 times faster on CPU and about 40 times faster on wall time.
- It still under-predicts, by about 26% on average (bias approx. -0.30), and half of the predictions miss by more than 47% of the true value.

#### `LightGBM`

We then train LightGBM, a gradient boosting library built on the same histogram idea, as our main model. It is fast on large tabular data, handles categorical columns natively, and grows trees leaf-wise, which often fits better than level-wise growth. 

To limit overfitting, we use:
- `learning_rate` of 0.05
- `leaves` of 31 
- `min_data_in_leaf` of 200
- Train for a maxinum of 2000 rounds, a log evaluation every 100 rounds, and stop when the RMSE stays stagnant for 50 rounds.

In [79]:
ds_train_no_hist = lgb.Dataset(
    df_train[FEATURES_WITHOUT_HISTORY],
    df_train[TARGET],
    categorical_feature=FEATURES_CAT,
    free_raw_data=False,
)
ds_val_no_hist = lgb.Dataset(
    df_val[FEATURES_WITHOUT_HISTORY],
    df_val[TARGET],
    categorical_feature=FEATURES_CAT,
    free_raw_data=False,
)

In [80]:
%%time
lgb_no_hist = lgb.train(
    dict(
        objective="regression",
        metric="rmse",
        learning_rate=0.05,
        num_leaves=31,
        min_data_in_leaf=200,
        feature_pre_filter=False,
    ),
    ds_train_no_hist,
    num_boost_round=2000,
    valid_sets=[ds_val_no_hist],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

evaluate(
    "val",
    "lgb-no-history",
    ds_val_no_hist,
    lgb_no_hist.predict(
        df_val[FEATURES_WITHOUT_HISTORY],
        num_iteration=lgb_no_hist.best_iteration,
    ),
)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.882817
[200]	valid_0's rmse: 0.856803
[300]	valid_0's rmse: 0.847559
[400]	valid_0's rmse: 0.843983
[500]	valid_0's rmse: 0.84177
[600]	valid_0's rmse: 0.839694
[700]	valid_0's rmse: 0.838193
[800]	valid_0's rmse: 0.83667
[900]	valid_0's rmse: 0.836041
[1000]	valid_0's rmse: 0.834848
[1100]	valid_0's rmse: 0.833979
[1200]	valid_0's rmse: 0.833668
[1300]	valid_0's rmse: 0.83336
[1400]	valid_0's rmse: 0.833276
[1500]	valid_0's rmse: 0.832803
[1600]	valid_0's rmse: 0.832674
Early stopping, best iteration is:
[1571]	valid_0's rmse: 0.832573

    Results (`val` split):
        Model name: lgb-no-history
        MedAPE %: 44.78
        Log RMSE: 0.8326
        Bias: -0.2833
    
CPU times: total: 1min 38s
Wall time: 15.9 s


**Observations**:

- The model beats the baseline. The MedAPE falls from 81.82% to 44.78%, and the log RMSE from 1.689 to 0.833, about 51% lower.
- It has the lowest error of the boosted models on every metric:
    - Its log RMSE of 0.833 is about 3% below `HistGradientBoostingRegressor` (0.859) and about 10% below `GradientBoostingRegressor` (0.927).
    - The validation RMSE improves quickly at first (0.883 at round 100, 0.848 at round 300) and then more slowly (0.842 at round 500, 0.833 at round 1,571). Later rounds still help, but only a little.
    - Training takes about 98 seconds of CPU time (16 seconds wall) for roughly 1,620 rounds (best round 1,571, plus 50 rounds of early stopping). Wall time is about 6 times `HistGradientBoostingRegressor` (2.7 seconds) and about 7 times faster than `GradientBoostingRegressor` (113 seconds). CPU time is close to `GradientBoostingRegressor`, and much higher than `HistGradientBoostingRegressor`.
    - It still under-predicts, by about 25% on average (bias approx. -0.28), and half of the predictions miss by more than 45% of the true value.
- Early stopping used the validation set, so its validation scores are slightly optimistic compared with the other models.
**Decision:** Use `LightGBM` as the main model without history. It has the lowest validation error on all three metrics. Its lead over `HistGradientBoostingRegressor` is modest (about 3% on log RMSE), and `HistGradientBoostingRegressor` is faster.

#### Recap

In [ ]:
evaluate("val", "dummy-median", df_val[TARGET], dummy.predict(df_val[FEATURES_NUM]))

evaluate(
    "val",
    "gb-no-history",
    df_val[TARGET],
    gb_no_hist.predict(df_val[FEATURES_WITHOUT_HISTORY]),
)

evaluate(
    "val",
    "hgb-no-history",
    df_val[TARGET],
    hgb_no_hist.predict(df_val[FEATURES_WITHOUT_HISTORY]),
)

evaluate(
    "val",
    "lgb-no-history",
    ds_val_no_hist,
    lgb_no_hist.predict(
        df_val[FEATURES_WITHOUT_HISTORY],
        num_iteration=lgb_no_hist.best_iteration,
    ),
)


    Results (`val` split):
        Model name: dummy-median
        MedAPE %: 81.82
        Log RMSE: 1.6890
        Bias: -0.5933
    

    Results (`val` split):
        Model name: gb-no-history
        MedAPE %: 52.63
        Log RMSE: 0.9271
        Bias: -0.3004
    

    Results (`val` split):
        Model name: hgb-no-history
        MedAPE %: 46.62
        Log RMSE: 0.8593
        Bias: -0.3033
    

    Results (`val` split):
        Model name: lgb-no-history
        MedAPE %: 44.78
        Log RMSE: 0.8326
        Bias: -0.2833
    


We summarise the models trained so far on the validation split. All of them use `FEATURES_WITHOUT_HISTORY` and are scored with the same three metrics. ``````LightGBM`````` is the run with the default squared-error loss. 

Times are CPU times for a single run. Best metrics (lower is better for MedAPE % and Log RMSE, lower absolute value is better for bias) are boldened.

Model |	MedAPE % | Log RMSE | Bias | Time (approx. CPU times) |
|---|:---:|:---:|:---:|:---:|
| `DummyRegressor` (median) | 81.82 | 1.6890 | -0.5933 | 31 ms |
| `GradientBoostingRegressor` | 52.63 | 0.9271 | -0.3004 | 102 s |
| `HistGradientBoostingRegressor` | 46.62 | 0.8593 | -0.3033 | 9 s |
| `LightGBM` | **44.78** | **0.8326** | **-0.2833** | 98 s |

**Observations**:

Observations:
- All three boosted models beat the baseline.
    - Log RMSE falls from 1.689 to between 0.833 and 0.927 (about 45% to 51% lower).
    - MedAPE falls from 81.82% to between 44.78% and 52.63%.
    - `LightGBM` is best on every metric, then `HistGradientBoostingRegressor`, then `GradientBoostingRegressor`. The log RMSE differs by 0.095 between the best and the worst (about 10%), and the MedAPE by 7.85 points. The gap between `LightGBM` and `HistGradientBoostingRegressor` is smaller: 0.027 on log RMSE (about 3%) and 1.8 points of MedAPE. The library matters more against classic `GradientBoostingRegressor` than between the two histogram-based models.
    - All models still under-predict, by about 25% to 26% on average (bias -0.28 to -0.30, against -0.59 for the baseline), and half of the predictions miss by more than 45% to 53% of the true value.
- Training time does not follow the accuracy order. `GradientBoostingRegressor` takes about 102 seconds of CPU time (113 seconds wall). `HistGradientBoostingRegressor` takes about 9 seconds of CPU time (2.7 seconds wall). `LightGBM` takes about 98 seconds of CPU time but only 16 seconds wall, so parallelism closes most of the gap with `GradientBoostingRegressor` on the clock. `HistGradientBoostingRegressor` is the fastest on both.

**Decision:** Use `LightGBM` as the main model without history, since it has the lowest error on all three metrics. The accuracy gap with `HistGradientBoostingRegressor` is modest, and that model is faster, so speed still counts. These scores (MedAPE 44.78%, log RMSE 0.833) are the ones the models with history need to beat.

### Value prediction models with previous valuations

#### Our baseline: Previous valuations only + median valuations for ones without

Now we evaluate models that incorporate previous valuations. We first establish a simple baseline using previous values with a median fallback.

In [82]:
has_previous_valuation = df_val["previous_valuation"].notna().to_numpy()
previous_valuations = df_val["previous_valuation"].to_numpy()

target_with_previous_valuation = df_val[TARGET]
prediction_with_previous_valuation = np.where(
    has_previous_valuation, previous_valuations, df_train[TARGET].median()
)

evaluate(
    "val",
    "previous-value-and-median-fallback",
    target_with_previous_valuation,
    prediction_with_previous_valuation,
)


    Results (`val` split):
        Model name: previous-value-and-median-fallback
        MedAPE %: 20.00
        Log RMSE: 0.5514
        Bias: 0.0284
    


**Observations:**
 - The baseline is much  stronger than the one without previous valuations, with a MedAPE of 20.00%, meaning for half of the players the prediction misses by less than 20% of the true value. This shows that past valuations have a big impact on determine the next one.
- The log RMSE is 0.5514, which translates to a typical error of a factor of about 1.7 in EUR, drastically reducing the spread seen in the previous baseline.
- The bias is near zero at 0.0284, indicating that previous valuations serve as a largely unbiased predictor when available, unlike the heavily skewed median predictor.

**Decision**: Use this as the benchmark for history-based models. The advanced model must beat:
- `MedAPE` of 20.00%
- `Log RMSE` of 0.5514

on the validation set.



#### ``HistGradientBoostingRegressor`` with previous valuations

We now trains and evaluates a ``HistGradientBoostingRegressor`` using the full feature set with historical data.

In [ ]:
for cat in FEATURES_CAT:
    dtype = pd.CategoricalDtype(sorted(df_train[cat].dropna().unique()))
    df_train[cat] = df_train[cat].astype(dtype)
    df_val[cat] = df_val[cat].astype(dtype)


hgb_hist = HistGradientBoostingRegressor(
    categorical_features="from_dtype",
    random_state=SEED,
)

In [110]:
%%time
hgb_hist.fit(df_train[FEATURES_WITH_HISTORY], df_train[TARGET])

evaluate(
    "val",
    "hgb-with-history",
    df_val[TARGET],
    hgb_hist.predict(df_val[FEATURES_WITH_HISTORY]),
)


    Results (`val` split):
        Model name: hgb-with-history
        MedAPE %: 14.29
        Log RMSE: 0.4232
        Bias: -0.0628
    
CPU times: total: 7.64 s
Wall time: 2.32 s


**Observations:**

- The model beats the carry-forward baseline. MedAPE falls from 20.00% to 14.29%, so for half of the players the error is now well below 20%.
- Log RMSE drops from 0.5514 to 0.4232, about 23% lower.
- The bias is -0.0628, about 6% too low on average. That is a mild underestimation, and much closer to zero than the models without history.

#### `LightGBM` with previous valuations

We now train a `LightGBM` model with the same hyperparameters as before, but utilizing the entire features set with previous valuations.

In [85]:
ds_train_with_hist = lgb.Dataset(
    df_train[FEATURES_WITH_HISTORY],
    df_train[TARGET],
    categorical_feature=FEATURES_CAT,
)
ds_val_with_hist = lgb.Dataset(
    df_val[FEATURES_WITH_HISTORY],
    df_val[TARGET],
    categorical_feature=FEATURES_CAT,
    reference=ds_train_with_hist,
)

In [86]:
%%time
lgb_hist = lgb.train(
    dict(
        objective="regression",
        metric="rmse",
        learning_rate=0.05,
        num_leaves=31,
        min_data_in_leaf=200,
        feature_pre_filter=False,
    ),
    ds_train_with_hist,
    num_boost_round=2000,
    valid_sets=[ds_val_with_hist],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

evaluate(
    "val",
    "lgb-with-history",
    ds_val_with_hist,
    lgb_hist.predict(
        df_val[FEATURES_WITH_HISTORY],
        num_iteration=lgb_hist.best_iteration,
    ),
)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.430325
[200]	valid_0's rmse: 0.42056
[300]	valid_0's rmse: 0.41799
[400]	valid_0's rmse: 0.416705
[500]	valid_0's rmse: 0.415814
[600]	valid_0's rmse: 0.414949
[700]	valid_0's rmse: 0.414361
Early stopping, best iteration is:
[652]	valid_0's rmse: 0.414317

    Results (`val` split):
        Model name: lgb-with-history
        MedAPE %: 14.05
        Log RMSE: 0.4143
        Bias: -0.0599
    
CPU times: total: 55.1 s
Wall time: 8.8 s


**Observations:**
- The model stops early at iteration 652, with a validation RMSE of 0.4143.
- It reaches a MedAPE of 14.05%, a log RMSE of 0.4143, and a bias of -0.0599.
- That is a slight improvement over HistGradientBoostingRegressor (14.29%, 0.4232, -0.0628) on every metric. The gaps are small.

#### Hybrid model: `lgb_hist` for rows with valuation history, `lgb_no_hist` for ones without

To address the distinct characteristics of players with and without historical data, this section implements a hybrid approach:
- Players with a previous valuation have a strong anchor, allowing their model to focus on predicting the nuanced changes from that baseline.
- Players without history lack this anchor, requiring a model that estimates their absolute value purely from their other attributes. 

By routing the data between `lgb_hist` and `lgb_no_hist` based on availability, each model can specialize in its specific task.

In [87]:
_prediction = pd.Series(np.nan, index=df_val.index)

_prediction.loc[has_previous_valuation] = lgb_hist.predict(
    df_val.loc[has_previous_valuation, FEATURES_WITH_HISTORY],
    num_iteration=lgb_hist.best_iteration,
)
_prediction.loc[~has_previous_valuation] = lgb_no_hist.predict(
    df_val.loc[~has_previous_valuation, FEATURES_WITHOUT_HISTORY],
    num_iteration=lgb_no_hist.best_iteration,
)

evaluate("val", "lgb-hybrid", df_val[TARGET], _prediction)


    Results (`val` split):
        Model name: lgb-hybrid
        MedAPE %: 14.03
        Log RMSE: 0.4058
        Bias: -0.0469
    


**Observations:**
- The hybrid model achieves a MedAPE of 14.03%, a log RMSE of 0.4058, and a bias of -0.0469.
- MedAPE is essentially unchanged from lgb-with-history (14.05%). Log RMSE improves from 0.4143 to 0.4058, about 2% lower.
- The bias moves from -0.0599 to -0.0469, closer to zero (about 5% too low, against about 6% for the single model).

#### `LightGBM` predicting valuation deltas

For players with historical data, predicting the delta is often easier than predicting the absolute value. We do that by train a `LightGBM` model to estimate how much a player's valuation will change from their last known figure.

Since this model predicts the change in valuation (target minus previous valuation), we filter both the training and validation sets to only include players who actually have a previous valuation. For players without one, the difference is undefined, so they are excluded from this specific model.

In [88]:
mask_train_with_prev = df_train["previous_valuation"].notna()
mask_val_with_prev = df_val["previous_valuation"].notna()

ds_train_delta = lgb.Dataset(
    df_train.loc[mask_train_with_prev, FEATURES_WITH_HISTORY],
    df_train.loc[mask_train_with_prev, TARGET]
    - df_train.loc[mask_train_with_prev, "previous_valuation"],
    categorical_feature=FEATURES_CAT,
)

ds_val_delta = lgb.Dataset(
    df_val.loc[mask_val_with_prev, FEATURES_WITH_HISTORY],
    df_val.loc[mask_val_with_prev, TARGET]
    - df_val.loc[mask_val_with_prev, "previous_valuation"],
    categorical_feature=FEATURES_CAT,
    reference=ds_train_delta,
)

In [89]:
%%time
lgb_delta_hist = lgb.train(
    dict(
        objective="regression",
        metric="rmse",
        learning_rate=0.05,
        num_leaves=31,
        min_data_in_leaf=200,
        feature_pre_filter=False,
    ),
    ds_train_delta,
    num_boost_round=2000,
    valid_sets=[ds_val_delta],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.283654
[200]	valid_0's rmse: 0.277015
[300]	valid_0's rmse: 0.275165
[400]	valid_0's rmse: 0.274201
[500]	valid_0's rmse: 0.273375
[600]	valid_0's rmse: 0.272977
[700]	valid_0's rmse: 0.272637
[800]	valid_0's rmse: 0.272427
Early stopping, best iteration is:
[801]	valid_0's rmse: 0.272421
CPU times: total: 53 s
Wall time: 7.64 s


Here we combine both models into a single prediction array. 
- `lgb_delta_hist` predicts the change for players with a baseline while 
- `lgb_no_hist` handles the rest. 

This ensures a fair evaluation where every row gets a prediction from the model trained for its data type, and the combined result is scored against the full validation set as `lgb-hybrid-delta`.

In [90]:
_prediction = pd.Series(np.nan, index=df_val.index)

_prediction.loc[has_previous_valuation] = df_val.loc[
    has_previous_valuation, "previous_valuation"
] + lgb_delta_hist.predict(
    df_val.loc[has_previous_valuation, FEATURES_WITH_HISTORY],
    num_iteration=lgb_delta_hist.best_iteration,
)
_prediction.loc[~has_previous_valuation] = lgb_no_hist.predict(
    df_val.loc[~has_previous_valuation, FEATURES_WITHOUT_HISTORY],
    num_iteration=lgb_no_hist.best_iteration,
)


evaluate("val", "lgb-hybrid-delta", df_val[TARGET], _prediction)


    Results (`val` split):
        Model name: lgb-hybrid-delta
        MedAPE %: 13.61
        Log RMSE: 0.4031
        Bias: -0.0437
    


**Observations:**

- The hybrid delta model achieves a MedAPE of 13.61%, a log RMSE of 0.4031, and a bias of -0.0437.
- This is a small but consistent improvement over `lgb-hybrid` (MedAPE 14.03%, log RMSE 0.4058, bias -0.0469). Most of the gain is on MedAPE.
- The bias stays small, at about 4% too low.

#### Recap

This is a recap of the findings. Lower is better for MedAPE and Log RMSE, lower absolute is better for bias.

| Model | MedAPE % | Log RMSE | Bias|
|---|:---:|:---:|:---:|
| `previous-value-and-median-fallback` | 20.00 | 0.5514 | **0.0284** |
| `hgb-with-history` | 14.29 | 0.4232 | -0.0628 |
| `lgb-with-history` | 14.05 | 0.4143 | -0.0599 |
| `lgb-hybrid` | 14.03 | 0.4058 | -0.0469 | 
| `lgb-hybrid-delta` | **13.61** | **0.4031** | -0.0437 |

**Observations**:

- Adding the previous valuation changes the problem. Carrying it forward, with a median fallback for first valuations, already gives a MedAPE of 20.00% and a log RMSE of 0.551, against about 45% and 0.83 for the best model without history.
- The boosted models improve on that baseline. HistGradientBoostingRegressor reaches 14.29% and 0.423, and LightGBM is slightly better (14.05% and 0.414). The gap between the two is small next to the gain from adding the history.
- The hybrid lowers the log RMSE from 0.414 to 0.406 and the bias from -0.060 to -0.047. Predicting the change instead of the level gives the lowest MedAPE (13.61%) and the lowest log RMSE (0.403), with a bias of -0.044. That is not the smallest bias: the carry-forward baseline is closer to zero (0.028), but its error is much larger.
- The under-prediction of the models without history (bias approx. -0.28) mostly disappears. The history models sit between about 4% and 6% too low.

**Decision:** Use the hybrid model that predicts the change as the main model with history. It has the lowest MedAPE and log RMSE. Its bias is small, though the carry-forward baseline is slightly less biased.

## Hyperparameter tuning

In [91]:
ds_train_delta = lgb.Dataset(
    df_train.loc[mask_train_with_prev, FEATURES_WITH_HISTORY],
    df_train.loc[mask_train_with_prev, TARGET]
    - df_train.loc[mask_train_with_prev, "previous_valuation"],
    categorical_feature=FEATURES_CAT,
    free_raw_data=False,
)
ds_val_delta = lgb.Dataset(
    df_val.loc[mask_val_with_prev, FEATURES_WITH_HISTORY],
    df_val.loc[mask_val_with_prev, TARGET]
    - df_val.loc[mask_val_with_prev, "previous_valuation"],
    categorical_feature=FEATURES_CAT,
    free_raw_data=False,
    reference=ds_train_delta,
)
ds_train_all = lgb.Dataset(
    df_train[FEATURES_WITHOUT_HISTORY],
    df_train[TARGET],
    categorical_feature=FEATURES_CAT,
    free_raw_data=False,
)
ds_val_all = lgb.Dataset(
    df_val[FEATURES_WITHOUT_HISTORY],
    df_val[TARGET],
    categorical_feature=FEATURES_CAT,
    free_raw_data=False,
    reference=ds_train_all,
)


def get_params(trial):
    return {
        "objective": "regression",
        "metric": "rmse",
        "verbose": -1,
        "feature_pre_filter": False,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 300),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
    }

In [92]:
def _objective(trial, dtrain, dval):
    model = lgb.train(
        get_params(trial),
        dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    trial.set_user_attr("best_iteration", model.best_iteration)
    return model.best_score["valid_0"]["rmse"]


def objective_lgb_delta(trial):
    return _objective(trial, ds_train_delta, ds_val_delta)


def objective_lgb_no_hist(trial):
    return _objective(trial, ds_train_all, ds_val_all)

### Tuning `lgb_delta`

In [93]:
study_delta = optuna.create_study(direction="minimize")
study_delta.optimize(objective_lgb_delta, n_trials=30)

[I 2026-11-13 14:17:33,466] A new study created in memory with name: no-name-dccc4549-870d-4bd5-bf10-047aa6e20e29
[I 2026-11-13 14:18:06,453] Trial 0 finished with value: 0.2730336326330124 and parameters: {'learning_rate': 0.01727164374577947, 'num_leaves': 119, 'min_data_in_leaf': 213, 'feature_fraction': 0.6661093873131652, 'bagging_fraction': 0.7562284533235484, 'bagging_freq': 2, 'lambda_l1': 0.012668453302846341, 'lambda_l2': 0.0016779316531597367}. Best is trial 0 with value: 0.2730336326330124.
[I 2026-11-13 14:18:14,279] Trial 1 finished with value: 0.2724726221585291 and parameters: {'learning_rate': 0.06949433067306462, 'num_leaves': 31, 'min_data_in_leaf': 135, 'feature_fraction': 0.7381082416437711, 'bagging_fraction': 0.9119079834742057, 'bagging_freq': 4, 'lambda_l1': 0.0003761034217074663, 'lambda_l2': 1.2911711667106704e-06}. Best is trial 1 with value: 0.2724726221585291.
[I 2026-11-13 14:18:55,387] Trial 2 finished with value: 0.2728882588087553 and parameters: {'lea

In [94]:
print(f"Best Delta Params: {study_delta.best_params}")
print(f"Best Delta Log RMSE: {study_delta.best_value:.4f}")

Best Delta Params: {'learning_rate': 0.0329357197051579, 'num_leaves': 84, 'min_data_in_leaf': 34, 'feature_fraction': 0.6638564410469625, 'bagging_fraction': 0.8996565474294665, 'bagging_freq': 3, 'lambda_l1': 0.2356497982814396, 'lambda_l2': 0.8290147259134233}
Best Delta Log RMSE: 0.2714


### Tuning `lgb_no_hist`

In [95]:
study_no_hist = optuna.create_study(direction="minimize")
study_no_hist.optimize(objective_lgb_no_hist, n_trials=30)

[I 2026-11-13 14:27:42,546] A new study created in memory with name: no-name-09eb5c99-8d30-49bb-8ed7-bd83d02f773d
[I 2026-11-13 14:28:37,251] Trial 0 finished with value: 0.833834547086711 and parameters: {'learning_rate': 0.014280716335715477, 'num_leaves': 81, 'min_data_in_leaf': 62, 'feature_fraction': 0.8416625633772223, 'bagging_fraction': 0.9052119665660474, 'bagging_freq': 5, 'lambda_l1': 1.959258772733247e-05, 'lambda_l2': 0.0011441010623128966}. Best is trial 0 with value: 0.833834547086711.
[I 2026-11-13 14:29:43,819] Trial 1 finished with value: 0.8322503704244665 and parameters: {'learning_rate': 0.01948444724058914, 'num_leaves': 105, 'min_data_in_leaf': 46, 'feature_fraction': 0.9658450721854144, 'bagging_fraction': 0.8567820173010341, 'bagging_freq': 9, 'lambda_l1': 1.3300444108443008, 'lambda_l2': 1.5897397106738387}. Best is trial 1 with value: 0.8322503704244665.
[I 2026-11-13 14:31:48,558] Trial 2 finished with value: 0.8330752794529779 and parameters: {'learning_rat

In [96]:
print(f"Best No-Hist Params: {study_no_hist.best_params}")
print(f"Best No-Hist Log RMSE: {study_no_hist.best_value:.4f}")

Best No-Hist Params: {'learning_rate': 0.026701613920489246, 'num_leaves': 85, 'min_data_in_leaf': 280, 'feature_fraction': 0.9705491626921462, 'bagging_fraction': 0.7054730035569816, 'bagging_freq': 2, 'lambda_l1': 7.845633287615977e-06, 'lambda_l2': 0.0008770828581963284}
Best No-Hist Log RMSE: 0.8301


### Tuning `lgb_hybrid_delta`

In [ ]:
best_delta_params = study_delta.best_params
best_no_hist_params = study_no_hist.best_params

tuned_lgb_delta = lgb.train(
    best_delta_params,
    ds_train_delta,
    num_boost_round=2000,
    valid_sets=[ds_val_delta],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

tuned_lgb_no_hist = lgb.train(
    best_no_hist_params,
    ds_train_no_hist,
    num_boost_round=2000,
    valid_sets=[ds_val_no_hist],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

In [98]:
_prediction = pd.Series(np.nan, index=df_val.index)

_prediction.loc[has_previous_valuation] = df_val.loc[
    has_previous_valuation, "previous_valuation"
] + tuned_lgb_delta.predict(
    df_val.loc[has_previous_valuation, FEATURES_WITH_HISTORY],
    num_iteration=tuned_lgb_delta.best_iteration,
)
_prediction.loc[~has_previous_valuation] = tuned_lgb_no_hist.predict(
    df_val.loc[~has_previous_valuation, FEATURES_WITHOUT_HISTORY],
    num_iteration=tuned_lgb_no_hist.best_iteration,
)

evaluate("val", "lgb-hybrid-delta-tuned", df_val[TARGET], _prediction)


    Results (`val` split):
        Model name: lgb-hybrid-delta-tuned
        MedAPE %: 13.60
        Log RMSE: 0.4015
        Bias: -0.0415
    


### Recap

This is a recap of the findings. Lower is better for MedAPE and Log RMSE, lower absolute is better for bias.

| Model | MedAPE % | Log RMSE | Bias |
|---|:---:|:---:|:---:|
| `lgb-hybrid-delta` | 13.61 | 0.4031 | -0.0437 |
| `lgb-hybrid-delta-tuned` (Optuna) | **13.60** | **0.4015** | **-0.0415** |

**Observations:**
- Tuning is a small improvement on every metric:
    - MedAPE is essentially unchanged (13.60% against 13.61%).
    - Log RMSE improves from 0.4031 to 0.4015 (about 0.4% lower).
    - Bias improves from -0.0437 to -0.0415. The tuned model is about 4% too low on average, against about 4.3% before.
    - The same pattern shows up in the tuning objective. The tuned delta model reaches 0.2714 RMSE on the change, against 0.2724 for the untuned run. The tuned no-history model reaches 0.8301, against 0.8326 before.
- The differences are small enough that the two hybrids are close to equivalent.

**Decision:** Keep the tuned hybrid as the final  model. It is slightly better on all three metrics. Early stopping and Optuna both used the validation set, so the edge is slightly optimistic, but it points the same way on every metric.

### Examine on test set

#### Pick the final model(s)

In [ ]:
final_lgb_delta = tuned_lgb_delta
final_lgb_no_hist = tuned_lgb_no_hist

#### Prepare the test dataset

In [120]:
for cat in FEATURES_CAT:
    dtype = pd.CategoricalDtype(sorted(df_train[cat].dropna().unique()))
    df_test[cat] = df_test[cat].astype(dtype)

mask_test_with_prev = df_test["previous_valuation"].notna()

test_prediction = pd.Series(np.nan, index=df_test.index)

predicted_delta_test = final_lgb_delta.predict(
    df_test.loc[mask_test_with_prev, FEATURES_WITH_HISTORY],
    num_iteration=final_lgb_delta.best_iteration,
)

test_prediction.loc[mask_test_with_prev] = (
    df_test.loc[mask_test_with_prev, "previous_valuation"].values + predicted_delta_test
)

prediction_no_hist_test = final_lgb_no_hist.predict(
    df_test.loc[~mask_test_with_prev, FEATURES_WITHOUT_HISTORY], 
    num_iteration=final_lgb_no_hist.best_iteration
)

test_prediction.loc[~mask_test_with_prev] = prediction_no_hist_test

#### Evaluate

In [119]:
evaluate(
    "test",
    "lgb-hybrid-delta-final",
    df_test[TARGET],
    test_prediction
)


    Results (`test` split):
        Model name: lgb-hybrid-delta-final
        MedAPE %: 14.10
        Log RMSE: 0.4387
        Bias: -0.0604
    


**Observations:**

- The final model generalizes well to the unseen test set. The MedAPE rises only slightly from 13.60% on the validation set to 14.10% on the test set, a difference of just 0.5 percentage points.
- The Log RMSE increases from 0.4015 (val) to 0.4387 (test). While there is a small drop in performance, it remains significantly stronger than the original baseline and confirms the model is not severely overfitting to the validation split.
- The bias drifts slightly more negative, moving from -0.0415 to -0.0604. The model is consistently conservative, under-predicting by roughly 6% on average across the test set.

#### Pick the final model(s)

In [ ]:
final_lgb_delta = tuned_lgb_delta
final_lgb_no_hist = tuned_lgb_no_hist

### Save findings

#### Save model weights

In [111]:
DIR_MODEL.mkdir(parents=True, exist_ok=True)

In [112]:
final_lgb_delta.save_model(DIR_MODEL / "lgb_delta.txt")
final_lgb_no_hist.save_model(DIR_MODEL / "lgb_no_hist.txt")

#### Save tuning hyperparameters

In [113]:
configs = {"delta_params": best_delta_params, "no_hist_params": best_no_hist_params}

with open(DIR_MODEL / "tuning.json", "w") as fout:
    json.dump(configs, fout, indent=4)

#### Save used features

In [114]:
features = {
    "features_with_history": FEATURES_WITH_HISTORY,
    "features_without_history": FEATURES_WITHOUT_HISTORY,
    "categories": {cat: df_train[cat].cat.categories.tolist() for cat in FEATURES_CAT},
}

with open(DIR_MODEL / "features.json", "w") as fout:
    json.dump(features, fout, indent=4)